In [ ]:
import pandas as pd
import numpy as np
import os
import json
from datetime import datetime

import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report
from sklearn.inspection import permutation_importance
from sklearn.neural_network import MLPClassifier
import joblib

%matplotlib inline

print("All libraries loaded.")

In [ ]:
df = pd.read_csv("Epilepsy_dataset.csv")

# Removes unessential columns
un_cols = [
    'EEG Abnormality Detected',
    'MRI/CT Scan Result',
    'Genetic Disorder',
    'Head Injury History',
    'Brain Tumor',
    'History of Stroke',
    'Developmental Delay (in Children)',
    'Weight',
    'Height',
    'Seizure Frequency',
    'Seizure Duration'
]

df_model = df.drop(columns=un_cols).copy()
print("Dataset loaded:", df_model.shape)

In [ ]:
binary_map = {
    'On Medication': 0, 'Not on Medication': 1,
    'Yes': 1, 'No': 0
}

binary_cols = [
    'Medication Status', 'Alcohol or Drug Use',
    'Aura Before Seizure', 'Loss of Consciousness', 'Muscle Stiffness',
    'Jerky Movements', 'Postictal Confusion', 'Blank Stare Episodes',
    'Eye Rolling', 'Stress or Anxiety Before Episode', 'Lack of Sleep Before Episode',
    'Flashing Lights Sensitivity', 'Loud Sound Sensitivity', 'Missed Medication',
    'Family History of Epilepsy'
]

for col in binary_cols:
    if col in df_model.columns:
        df_model[col] = df_model[col].map(binary_map)
    else:
        print(f"{col} not found in df_model. Skipping binary mapping.")

categorical_cols = ['Gender', 'Seizure Type', 'Target/Epilepsy Type']
le_map = {}

for col in categorical_cols:
    if col in df_model.columns:
        le = LabelEncoder()
        df_model[col] = le.fit_transform(df_model[col])
        le_map[col] = le
    else:
        raise ValueError(f"{col} not found in dataset.")

In [ ]:
df_model["Risk_Score"] = (
    df_model["Aura Before Seizure"] +
    df_model["Loss of Consciousness"] +
    df_model["Jerky Movements"] +
    df_model["Blank Stare Episodes"] +
    df_model["Postictal Confusion"] +
    df_model["Muscle Stiffness"] +
    df_model["Stress or Anxiety Before Episode"] +
    df_model["Lack of Sleep Before Episode"] +
    df_model["Flashing Lights Sensitivity"] +
    df_model["Loud Sound Sensitivity"] +
    df_model["Missed Medication"] +
    df_model["Alcohol or Drug Use"]
)

df_model["Seizure_Risk"] = pd.cut(
    df_model["Risk_Score"],
    bins=[-1, 2, 5, 20],
    labels=["Low", "Moderate", "High"]
)

print(df_model["Seizure_Risk"].value_counts())

le_target = LabelEncoder()
y_enc = le_target.fit_transform(df_model["Seizure_Risk"])

In [ ]:
exclude_cols = ["Seizure_Risk", "Risk_Score"]

feature_cols = [c for c in df_model.columns if c not in exclude_cols]

X = df_model[feature_cols]
y = y_enc

print("Training features:", feature_cols)
print("X:", X.shape, "y:", y.shape)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

rf = RandomForestClassifier(
    n_estimators=800,
    class_weight='balanced',
    random_state=42
)
rf.fit(X_train, y_train)

mlp = MLPClassifier(
    hidden_layer_sizes=(64, 32),
    max_iter=500,
    random_state=42
)
mlp.fit(X_train, y_train)

rf_pred = rf.predict(X_test)
mlp_pred = mlp.predict(X_test)

print("RF Accuracy:", accuracy_score(y_test, rf_pred))
print("\nMLP Accuracy:", accuracy_score(y_test, mlp_pred))

y_pred_ensemble = [
    np.bincount([rf_pred[i], mlp_pred[i]]).argmax()
    for i in range(len(rf_pred))
]

print("\nEnsemble Accuracy:", accuracy_score(y_test, y_pred_ensemble))
print("\nClassification Report:")
print(classification_report(y_test, y_pred_ensemble))

In [ ]:
importances = rf.feature_importances_

feat_df = pd.DataFrame({
    "Feature": X.columns,
    "Importance": importances
}).sort_values(by="Importance", ascending=False)

print("Top 15 Most Important Features (Random Forest):\n", feat_df.head(15))

plt.figure(figsize=(10, 7))
plt.barh(feat_df["Feature"].head(15), feat_df["Importance"].head(15))
plt.gca().invert_yaxis()
plt.title("Top 15 Most Important Features (Random Forest)")
plt.show()

In [ ]:
perm_result = permutation_importance(
    rf,
    X_test,
    y_test,
    n_repeats=10,
    random_state=42
)

perm_df = pd.DataFrame({
    "Feature": X.columns,
    "Importance": perm_result.importances_mean,
    "Standard": perm_result.importances_std
}).sort_values(by="Importance", ascending=False)

print("Top 15 Most Important Features (Permutation Importance):\n")
print(perm_df.head(15))

plt.figure(figsize=(10, 6))
plt.barh(perm_df["Feature"].head(15), perm_df["Importance"].head(15))
plt.gca().invert_yaxis()
plt.title("Top 15 Most Important Features (Permutation Importance)")
plt.show()

In [ ]:
joblib.dump(rf, "kasper_rf_risk_model.pkl")
joblib.dump(mlp, "kasper_mlp_risk_model.pkl")

print("Risk models saved.")

In [ ]:
SYMPTOM_MAP = {
    1: "Jerky Movements",
    2: "Eye Rolling",
    3: "Loss of Consciousness",
    4: "Blank Stare Episodes",
    5: "Postictal Confusion",
    6: "Muscle Stiffness"
}

TRIGGER_MAP = {
    1: "Flashing Lights Sensitivity",
    2: "Loud Sound Sensitivity",
    3: "Alcohol or Drug Use"
}

def kasper_setup():
    print("\nKASPER AI Seizure Log\n")
    print("To get started, please answer a few questions.")

    user = {}
    user["Name"] = input("What's your name?: ").strip()
    user["Age"] = int(input("How old are you?: "))
    user["Gender"] = input("What gender do you identify as? (Male/Female/Other): ").capitalize()
    user["Epilepsy_Type"] = input("What type of epilepsy were you diagnosed with? (Focal/Generalized/Absence/Not Confirmed/Normal): ").capitalize()
    user["On_Medication"] = input("Are you currently taking anti-seizure medication? (Yes/No): ").capitalize()
    user["Family_History"] = input("Do you have a family history of epilepsy? (Yes/No): ").capitalize()
    user["Baseline_Aura"] = input("Do you typically experience auras before seizures? (Yes/No): ").capitalize()

    print("\nPlease select any known symptoms you typically experience. If you're not sure, press Enter.")
    for num, label in SYMPTOM_MAP.items():
        print(f"{num} = {label}")
    sym_nums = input("Enter numbers separated by commas: ").strip()
    if sym_nums:
        user["Typical_Symptoms"] = [int(n.strip()) for n in sym_nums.split(",")]
    else:
        user["Typical_Symptoms"] = []

    print("\nPlease select any known triggers you have. If you're not sure, press Enter.")
    for num, label in TRIGGER_MAP.items():
        print(f"{num} = {label}")
    trig_nums = input("Enter numbers separated by commas: ").strip()
    if trig_nums:
        user["Known_Triggers"] = [int(n.strip()) for n in trig_nums.split(",")]
    else:
        user["Known_Triggers"] = []

    with open("kasper_user.json", "w") as f:
        json.dump(user, f, indent=4)

    print("\nSetup complete! KASPER will use this data automatically each log.\n")
    return user

In [ ]:
def kasper_daily_log(user):
    print(f"\nHello, {user['Name']}. How are you doing today?\n")
    
    log = {}

    log["Date"] = datetime.now().strftime("%Y-%m-%d")

    log["Aura_Today"] = input("Did you experience an aura today? (Yes/No): ").capitalize()
    
    log["Aura_Type"] = "0"
    log["Seizure_Today"] = "No"
    
    if log["Aura_Today"] == "Yes":
        seizure_after = input(
            "Did you have a seizure right after the aura? (Yes/No): "
        ).capitalize()

        if seizure_after == "Yes":
            log["Aura_Type"] = "1"  # Aura before seizure
            log["Seizure_Today"] = "Yes"
        else:
            log["Aura_Type"] = "2"  # Aura without seizure
            log["Seizure_Today"] = input(
                "Did you have a seizure at ANY other time today? (Yes/No): "
            ).capitalize()
    else:
        log["Seizure_Today"] = input("Did you have a seizure today? (Yes/No): ").capitalize()

    log["Stress_Level"] = input(
        "How stressed did you feel today? (Low/Moderate/High): "
    ).capitalize()
    log["Sleep_Quality"] = input(
        "How was your sleep last night? (Good/Fair/Poor): "
    ).capitalize()

    log["Missed_Medication"] = input(
        "Did you miss any medication today? (Yes/No): "
    ).capitalize()

    log["Flashing_Lights_Today"] = input("Were flashing lights uncomfortable today? (Yes/No): ").capitalize()
    log["Loud_Sounds_Today"] = input("Were loud sounds uncomfortable today? (Yes/No): ").capitalize()
    log["Alcohol_Use_Today"] = input("Did you ingest any alcohol or recreational drugs today? (Yes/No): ").capitalize()

    print("\nWhich symptoms did you experience today? If unsure, press Enter")
    for num, label in SYMPTOM_MAP.items():
        print(f"{num} = {label}")
    sym_today = input("Enter numbers separated by commas: ").strip()
    if sym_today:
        log["Symptoms_Today"] = [int(n.strip()) for n in sym_today.split(",")]
    else:
        log["Symptoms_Today"] = []

    print("\nWhich of these do you feel were triggers today? If unsure, press Enter")
    for num, label in TRIGGER_MAP.items():
        print(f"{num} = {label}")
    trig_today = input("Enter numbers separated by commas: ").strip()
    if trig_today:
        log["Triggers_Today"] = [int(n.strip()) for n in trig_today.split(",")]
    else:
        log["Triggers_Today"] = []

    return log

In [ ]:
# Builds a single row dictionary aligned with X.columns using the user's profile and the daily log
def build_model_row(user, log):
    row = {col: 0 for col in X.columns}

    row["Age"] = int(user["Age"])
    
    try:
        row["Gender"] = le_map["Gender"].transform([user["Gender"]])[0]
    except:
        row["Gender"] = 0

    try:
        row["Seizure Type"] = le_map["Seizure Type"].transform(
            [user["Epilepsy_Type"]]
        )[0]
    except:
        row["Seizure Type"] = 0

    try:
        row["Target/Epilepsy Type"] = le_map["Target/Epilepsy Type"].transform([user["Epilepsy_Type"]])[0]
    except:
        row["Target/Epilepsy Type"] = 0

    row["Medication Status"] = 0 if user["On_Medication"] == "Yes" else 1
    row["Family History of Epilepsy"] = 1 if user["Family_History"] == "Yes" else 0

    seizure_today = (log["Seizure_Today"] == "Yes")

    if seizure_today and log["Stress_Level"] == "High":
        row["Stress or Anxiety Before Episode"] = 1
    else:
        row["Stress or Anxiety Before Episode"] = 0

    if seizure_today and log["Sleep_Quality"] == "Poor":
        row["Lack of Sleep Before Episode"] = 1
    else:
        row["Lack of Sleep Before Episode"] = 0

    row["Flashing Lights Sensitivity"] = 1 if (
        seizure_today and log["Flashing_Lights_Today"] == "Yes"
    ) else 0

    row["Loud Sound Sensitivity"] = 1 if (
        seizure_today and log["Loud_Sounds_Today"] == "Yes"
    ) else 0

    row["Alcohol or Drug Use"] = 1 if (
        seizure_today and log["Alcohol_Use_Today"] == "Yes"
    ) else 0

    row["Missed Medication"] = 1 if (
        seizure_today and log["Missed_Medication"] == "Yes"
    ) else 0

    for name in SYMPTOM_MAP.values():
        if name in row:
            row[name] = 0

    for s in log["Symptoms_Today"]:
        name = SYMPTOM_MAP.get(s)
        if name in row:
            row[name] = 1

    row["Aura Before Seizure"] = 1 if log["Aura_Type"] == "1" else 0

    return row

In [ ]:
def kasper_risk(model_row):
    arr = pd.DataFrame([model_row], columns=X.columns)
    proba = rf.predict_proba(arr)[0]
    pred_idx = np.argmax(proba)
    pred = le_target.inverse_transform([pred_idx])[0]
    
    return proba[pred_idx], pred

In [ ]:
def ensemble_predict(model_row):
    arr = pd.DataFrame([model_row], columns=X.columns)
    rf_p = rf.predict_proba(arr)
    mlp_p = mlp.predict_proba(arr)
    avg = (rf_p + mlp_p) / 2
    pred_idx = np.argmax(avg)
    pred_type = le_target.inverse_transform([pred_idx])[0]

    return pred_type, float(np.max(avg))

In [ ]:
def save_log(user, log, model_row, rf_pred, rf_score, mlp_pred, mlp_conf, ensemble_type, ensemble_conf):
    row = {
        "Date": log["Date"],
        "Name": user["Name"],
        "Seizure_Today": log["Seizure_Today"],
        "Aura_Today": log["Aura_Today"],
        "Aura_Type": log["Aura_Type"],
        "Stress_Level": log["Stress_Level"],
        "Sleep_Quality": log["Sleep_Quality"],
        "Missed_Medication": log["Missed_Medication"],
        "Flashing_Lights_Today": log["Flashing_Lights_Today"],
        "Loud_Sounds_Today": log["Loud_Sounds_Today"],
        "Alcohol_Use_Today": log["Alcohol_Use_Today"],
        
        "RF_Predicted_Type": rf_pred,
        "RF_Confidence": round(rf_score, 4),

        "MLP_Predicted_Type": mlp_pred,
        "MLP_Confidence": round(mlp_conf, 4),

        "Ensemble_Predicted_Type": ensemble_type,
        "Ensemble_Confidence": round(ensemble_conf, 4)
    }

    row["Flashing_Lights_Exposed"] = 1 if log["Flashing_Lights_Today"] == "Yes" else 0
    row["Loud_Sound_Exposed"]      = 1 if log["Loud_Sounds_Today"] == "Yes" else 0
    row["Alcohol_Use_Today"]       = 1 if log["Alcohol_Use_Today"] == "Yes" else 0
    row["Missed_Medication_Flag"]  = 1 if log["Missed_Medication"] == "Yes" else 0
    row["High_Stress"]             = 1 if log["Stress_Level"] == "High" else 0
    row["Poor_Sleep"]              = 1 if log["Sleep_Quality"] == "Poor" else 0

    for name in SYMPTOM_MAP.values():
        row[f"Symptom_{name.replace(' ', '_')}"] = 1 if name in model_row and model_row[name] == 1 else 0

    df_row = pd.DataFrame([row])

    if not os.path.exists("kasper_logs.csv"):
        df_row.to_csv("kasper_logs.csv", index=False)
    else:
        df_row.to_csv("kasper_logs.csv", mode="a", index=False, header=False)

In [ ]:
def analyze_patterns():
    if not os.path.exists("kasper_logs.csv"):
        return ["I don't have any history yet. Keep logging and I'll start spotting patterns for you."]

    logs = pd.read_csv("kasper_logs.csv")
    if logs.shape[0] < 5:
        return ["You've logged a few days so far. With more entries, I'll be able to give stronger pattern insights."]

    logs["Seizure_Flag"] = logs["Seizure_Today"].str.capitalize().eq("Yes")
    logs["Aura_Flag"] = logs["Aura_Today"].str.capitalize().eq("Yes")

    total_days = len(logs)
    total_seizure_days = logs["Seizure_Flag"].sum()
    total_aura_days = logs["Aura_Flag"].sum()

    insights = []

    cond_cols = {
        "Flashing_Lights_Exposed": "flashing lights",
        "Loud_Sound_Exposed": "loud sounds",
        "Alcohol_Use_Today": "alcohol or drug use",
        "Missed_Medication_Flag": "missed medication",
        "High_Stress": "high stress",
        "Poor_Sleep": "poor sleep"
    }

    for col, label in cond_cols.items():
        if col not in logs.columns:
            continue

        cond_days = logs[logs[col] == 1]
        n_cond_days = len(cond_days)
        if n_cond_days == 0:
            continue

        seizure_with_cond = cond_days["Seizure_Flag"].sum()
        aura_with_cond = cond_days["Aura_Flag"].sum()

        if total_seizure_days > 0:
            seizure_ratio = seizure_with_cond / total_seizure_days
        else:
            seizure_ratio = 0

        if total_aura_days > 0:
            aura_ratio = aura_with_cond / total_aura_days
        else:
            aura_ratio = 0

        if total_seizure_days >= 2 and seizure_with_cond >= 1 and seizure_ratio >= 0.5:
            insights.append(
                f"{label.title()} has shown up on {n_cond_days} of your logged days, "
                f"and about {seizure_ratio*100:.0f}% of your seizures happened on those days. "
                "That might be a personal risk factor for you."
            )
        elif seizure_with_cond == 0 and total_seizure_days >= 2:
            insights.append(
                f"You've had {n_cond_days} day(s) with {label}, but it hasn’t lined up with seizure days so far. "
                "It doesn’t really look like a strong trigger in your logs yet."
            )

        if total_aura_days >= 2 and aura_with_cond >= 1 and aura_ratio >= 0.5:
            insights.append(
                f"{label.title()} also shows up a lot on days when you report auras "
                f"({aura_ratio*100:.0f}% of aura days). "
                "That could be something to keep an eye on."
            )

    if not insights:
        insights.append(
            "Right now, I’m not seeing any strong, consistent patterns between triggers and your seizures in the logs.\n"
        )

    return insights

In [ ]:
if os.path.exists("kasper_user.json"):
    with open("kasper_user.json", "r") as f:
        user = json.load(f)
else:
    user = kasper_setup()

log = kasper_daily_log(user)
model_row = build_model_row(user, log)

rf_score, rf_pred = kasper_risk(model_row)

mlp_proba = mlp.predict_proba(pd.DataFrame([model_row], columns=X.columns))[0]
mlp_pred = le_target.inverse_transform([np.argmax(mlp_proba)])[0]
mlp_conf = float(np.max(mlp_proba))

ensemble_type, ensemble_conf = ensemble_predict(model_row)

save_log(
    user, log, model_row,
    rf_pred, rf_score,
    mlp_pred, mlp_conf,
    ensemble_type, ensemble_conf
)

print("\nDaily log saved successfully!")
print("Analyzing your patterns...")

insights = analyze_patterns()

print("\nKASPER's Feedback:")
for i in insights:
    print("• " + i)